In [9]:
# Imports and Load Raw Data 

import pandas as pd
import numpy as np
from pathlib import Path

raw_df = pd.read_parquet(
    "Data/raw/Census/census_raw_all_geographies.parquet"
)

print("Raw data loaded")
print(f"   Rows    : {len(raw_df)}")
print(f"   Columns : {len(raw_df.columns)}")
print(f"\nGeographies and years:")
for geo in raw_df["geography"].unique():
    years = sorted(raw_df[raw_df["geography"]==geo]["acs_year"].unique())
    if len(years) > 3:
        print(f"   {geo}: {years[0]} → {years[-1]} ({len(years)} years)")
    else:
        print(f"   {geo}: {years}")

Raw data loaded
   Rows    : 45
   Columns : 51

Geographies and years:
   Tehama County: 2010 → 2024 (15 years)
   California: 2010 → 2024 (15 years)
   United States: 2010 → 2024 (15 years)


In [10]:
# Separate Geographies 

tehama = raw_df[
    raw_df["geography"] == "Tehama County"
].copy().sort_values("acs_year").reset_index(drop=True)

ca = raw_df[
    raw_df["geography"] == "California"
].copy().sort_values("acs_year").reset_index(drop=True)

us = raw_df[
    raw_df["geography"] == "United States"
].copy().sort_values("acs_year").reset_index(drop=True)

print("Geographies separated")
print(f"   Tehama : {len(tehama)} rows "
      f"({tehama['acs_year'].min()} → {tehama['acs_year'].max()})")
print(f"   CA     : {len(ca)} rows "
      f"({ca['acs_year'].min()} → {ca['acs_year'].max()})")
print(f"   US     : {len(us)} rows "
      f"({us['acs_year'].min()} → {us['acs_year'].max()})")

Geographies separated
   Tehama : 15 rows (2010 → 2024)
   CA     : 15 rows (2010 → 2024)
   US     : 15 rows (2010 → 2024)


In [22]:
# Calculate All Rates 

def calculate_rates(df):
    """
    Calculate all rates and percentages from raw counts.
    Works for any geography and any year.
    """
    d   = df.copy()
    pop = d["total_population"]

    # Age & Sex 
    d["male_pct"]   = (d["male_population"] / pop * 100).round(1)
    d["female_pct"] = (d["female_population"] / pop * 100).round(1)

    # Age bracket: 0-4
    d["pop_0_to_4"] = (
        d["male_under_5"] + d["female_under_5"]
    )
    d["pop_0_to_4_pct"] = (d["pop_0_to_4"] / pop * 100).round(1)

    # Age bracket: 5-17
    d["pop_5_to_17"] = (
        d["male_5_to_9"]   + d["female_5_to_9"]   +
        d["male_10_to_14"] + d["female_10_to_14"] +
        d["male_15_to_17"] + d["female_15_to_17"]
    )
    d["pop_5_to_17_pct"] = (d["pop_5_to_17"] / pop * 100).round(1)

    # Age bracket: 65-84
    d["pop_65_to_84"] = (
        d["male_65_66"]   + d["female_65_66"]  +
        d["male_67_69"]   + d["female_67_69"]  +
        d["male_70_74"]   + d["female_70_74"]  +
        d["male_75_79"]   + d["female_75_79"]  +
        d["male_80_84"]   + d["female_80_84"]
    )
    d["pop_65_to_84_pct"] = (d["pop_65_to_84"] / pop * 100).round(1)

    # Age bracket: 85+
    d["pop_85_plus"] = (
        d["male_85_plus"] + d["female_85_plus"]
    )
    d["pop_85_plus_pct"] = (d["pop_85_plus"] / pop * 100).round(1)

    # Age bracket: 18-64 (working age)
    # = total - (0-4) - (5-17) - (65-84) - (85+)
    d["pop_18_to_64"] = (
        pop -
        d["pop_0_to_4"] -
        d["pop_5_to_17"] -
        d["pop_65_to_84"] -
        d["pop_85_plus"]
    )
    d["pop_18_to_64_pct"] = (
        d["pop_18_to_64"] / pop * 100
    ).round(1)

    # Race & Ethnicity
    d["hispanic_pct"]    = (d["hispanic_latino"] / pop * 100).round(1)
    d["white_pct"]       = (d["white_alone"]     / pop * 100).round(1)
    d["black_pct"]       = (d["black"]           / pop * 100).round(1)
    d["aian_pct"]        = (d["american indian_alone"] / pop * 100).round(1)
    d["asian_pct"]       = (d["asian_alone"]     / pop * 100).round(1)
    d["nhpi_pct"]        = (d["nhpi_alone"]      / pop * 100).round(1)
    d["two_or_more_pct"] = (d["two_or_more_races"]/ pop * 100).round(1)

    # Income & Poverty 
    d["poverty_rate_pct"] = (
        d["population_below_poverty"] /
        d["poverty_universe"] * 100
    ).round(1)

    # Housing 
    d["homeownership_rate_pct"] = (
        d["owner_occupied_units"] /
        d["occupied_housing_units"] * 100
    ).round(1)
    d["renter_rate_pct"] = (
        100 - d["homeownership_rate_pct"]
    ).round(1)

    return d


print("calculate_rates() defined")
print("\nWill calculate:")
print("  Age brackets  : 0-4, 5-17, 18-64, 65+, 85+")
print("  Sex           : male_pct, female_pct")
print("  Race          : hispanic, white, black,")
print("                  aian, asian, nhpi,")
print("                  two_or_more")
print("  Income/Poverty: poverty_rate_pct")
print("  Housing       : homeownership_rate_pct,")
print("                  renter_rate_pct")

calculate_rates() defined

Will calculate:
  Age brackets  : 0-4, 5-17, 18-64, 65+, 85+
  Sex           : male_pct, female_pct
  Race          : hispanic, white, black,
                  aian, asian, nhpi,
                  two_or_more
  Income/Poverty: poverty_rate_pct
  Housing       : homeownership_rate_pct,
                  renter_rate_pct


In [23]:
# Apply Rates to All Geographies 

tehama_calc = calculate_rates(tehama)
ca_calc     = calculate_rates(ca)
us_calc     = calculate_rates(us)

print("Rates calculated for all geographies")
print(f"\nTehama  : {len(tehama_calc)} rows, "
      f"{len(tehama_calc.columns)} columns")
print(f"CA      : {len(ca_calc)} rows, "
      f"{len(ca_calc.columns)} columns")
print(f"US      : {len(us_calc)} rows, "
      f"{len(us_calc.columns)} columns")

print("\nSample — Tehama County 2024:")
print(f"{'Indicator':<25} {'Count':>10} {'Rate':>8}")
print("-" * 45)
t2024 = tehama_calc[tehama_calc["acs_year"] == 2024].iloc[0]

samples = [
    ("Total Population",  "total_population",      None),
    ("Male",              "male_population",        "male_pct"),
    ("Female",            "female_population",      "female_pct"),
    ("Age 0-4",           "pop_0_to_4",             "pop_0_to_4_pct"),
    ("Age 5-17",          "pop_5_to_17",            "pop_5_to_17_pct"),
    ("Age 18-64",         "pop_18_to_64",           "pop_18_to_64_pct"),
    ("Age 65-84",         "pop_65_to_84",            "pop_65_to_84_pct"),
    ("Age 85+",           "pop_85_plus",            "pop_85_plus_pct"),
    ("Hispanic",          "hispanic_latino",        "hispanic_pct"),
    ("White",             "white_alone",            "white_pct"),
    ("Poverty",           "population_below_poverty","poverty_rate_pct"),
    ("Homeowners",        "owner_occupied_units",   "homeownership_rate_pct"),
    ("Renters",           None,                     "renter_rate_pct"),
]

for label, count_col, rate_col in samples:
    count = f"{t2024[count_col]:,.0f}" if count_col else "—"
    rate  = f"{t2024[rate_col]}%"      if rate_col  else "—"
    print(f"{label:<25} {count:>10} {rate:>8}")

Rates calculated for all geographies

Tehama  : 15 rows, 73 columns
CA      : 15 rows, 73 columns
US      : 15 rows, 73 columns

Sample — Tehama County 2024:
Indicator                      Count     Rate
---------------------------------------------
Total Population              65,167        —
Male                          32,404    49.7%
Female                        32,763    50.3%
Age 0-4                        3,756     5.8%
Age 5-17                      11,940    18.3%
Age 18-64                     36,386    55.8%
Age 65-84                     11,642    17.9%
Age 85+                        1,443     2.2%
Hispanic                      18,941    29.1%
White                         43,477    66.7%
Poverty                       10,025    15.6%
Homeowners                    16,640    67.1%
Renters                            —    32.9%


In [25]:
# Combine and Save Processed Data 

from pathlib import Path

# Combine all three geographies
processed_df = pd.concat(
    [tehama_calc, ca_calc, us_calc],
    ignore_index=True
).sort_values(
    ["geography", "acs_year"]
).reset_index(drop=True)

print("All geographies combined")
print(f"\nProcessed dataset:")
print(f"   Rows    : {len(processed_df)}")
print(f"   Columns : {len(processed_df.columns)}")
print(f"\nRows per geography:")
for geo in processed_df["geography"].unique():
    n = len(processed_df[processed_df["geography"]==geo])
    print(f"   {geo}: {n} rows")

# Save processed file 
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = PROCESSED_DIR / "census_processed.parquet"
csv_path     = PROCESSED_DIR / "census_processed.csv"

processed_df.to_parquet(parquet_path, index=False)
processed_df.to_csv(csv_path, index=False)

print(f"\nParquet saved : {parquet_path}")
print(f"\nCSV saved     : {csv_path}")

All geographies combined

Processed dataset:
   Rows    : 45
   Columns : 73

Rows per geography:
   California: 15 rows
   Tehama County: 15 rows
   United States: 15 rows

Parquet saved : data\processed\census_processed.parquet

CSV saved     : data\processed\census_processed.csv


In [26]:
df= pd.read_parquet("Data/processed/census_processed.parquet")
df

,NAME,total_population,male_population,female_population,median_age,male_under_5,female_under_5,male_5_to_9,male_10_to_14,male_15_to_17,...,hispanic_pct,white_pct,black_pct,aian_pct,asian_pct,nhpi_pct,two_or_more_pct,poverty_rate_pct,homeownership_rate_pct,renter_rate_pct
0,California,36637290,18223157,18414133,34.9,1300849,1244216,1265861,1339624,867657,...,36.7,61.1,6.1,0.8,13.0,0.4,3.8,13.7,57.4,42.6
1,California,36969200,18387718,18581482,35.1,1301624,1243600,1271144,1327420,862762,...,37.2,61.8,6.1,0.8,13.1,0.4,3.9,14.4,56.7,43.3
2,California,37325068,18561020,18764048,35.2,1300557,1243220,1278085,1319653,851857,...,37.6,62.3,6.0,0.8,13.2,0.4,4.1,15.3,56.0,44.0
3,California,37659181,18726468,18932713,35.4,1291918,1235834,1284613,1313817,838167,...,37.9,62.3,6.0,0.8,13.3,0.4,4.3,15.9,55.3,44.7
4,California,38066920,18911519,19155401,35.6,1288128,1233171,1291849,1304704,824086,...,38.2,62.1,5.9,0.8,13.5,0.4,4.5,16.4,54.8,45.2
5,California,38421464,19087135,19334329,35.8,1283763,1228013,1295030,1297819,811114,...,38.4,61.8,5.9,0.7,13.7,0.4,4.5,16.3,54.3,45.7
6,California,38654206,19200970,19453236,36.0,1277459,1222102,1295679,1295400,800330,...,38.6,61.3,5.9,0.7,13.9,0.4,4.6,15.8,54.1,45.9
7,California,38982847,19366579,19616268,36.1,1275266,1218279,1290421,1297468,793298,...,38.8,60.6,5.8,0.7,14.1,0.4,4.7,15.1,54.5,45.5
8,California,39148760,19453769,19694991,36.3,1268862,1211817,1274390,1307143,785629,...,38.9,60.1,5.8,0.8,14.3,0.4,4.8,14.3,54.6,45.4
9,California,39283497,19526298,19757199,36.5,1254607,1196921,1257974,1318355,779960,...,39.0,59.7,5.8,0.8,14.5,0.4,4.9,13.4,54.8,45.2
